# Unit 04: Dot Products, Norms, and Projection

This unit is about alignment. Dot products measure directional agreement, norms measure length, cosine similarity removes magnitude, and projection gives the part of one vector that lies along another.

## Review Focus

Keep these ideas in view while working:

- Dot product: how much two vectors point in the same direction.
- Orthogonal vectors: dot product equals zero.
- Norm: vector length.
- Cosine similarity: alignment after normalizing away magnitude.
- Projection: the shadow of one vector onto another.
- Attention bridge: attention scores are dot products between query and key vectors.

## Core Formulas

Use these as references while solving. The point is not to memorize syntax; it is to connect the formula to the geometry.

```text
dot(a, b) = sum_i a_i * b_i

||a|| = sqrt(dot(a, a))

cosine(a, b) = dot(a, b) / (||a|| * ||b||)

project v onto u = (dot(u, v) / dot(u, u)) * u
```

Projection notation: `v` is the vector being projected. `u` is the direction you are projecting onto.

## By-Hand Problems

Write these out before coding. Use the PyTorch section afterward to check your answers.

### Problem 1: Dot product

Compute `u dot v` for:

```text
u = (1, 2, 3)
v = (4, -1, 0)
```

Then write one sentence about what the result says about alignment.

### Problem 2: Orthogonal vector

Find a vector orthogonal to both:

```text
(1, 1, 0)
(0, 1, 1)
```

Check it by confirming both dot products are zero.

### Problem 3: Project onto the x-axis

Project `(3, 4)` onto `(1, 0)`.

Geometric question: what part of `(3, 4)` survives when you project onto the x-axis?

### Problem 4: Project onto a diagonal unit vector

Project `(3, 4)` onto `(1, 1) / sqrt(2)`.

Geometric question: how is this different from projecting onto `(1, 0)`?

### Problem 5: Explain the geometric difference

Compare the two projections:

- Projection of `(3, 4)` onto `(1, 0)`
- Projection of `(3, 4)` onto `(1, 1) / sqrt(2)`

Write a few sentences explaining what each projection keeps and what each projection discards.

## PyTorch Practice: Fill the TODOs

The test vectors reuse your by-hand problem so you can sanity-check against numbers you already know. Fill in the four TODOs. Do not use the PyTorch built-ins inside your manual implementations unless the prompt says to use them for verification.

In [2]:
import torch

# Test vectors (reuse your by-hand problems so you can sanity-check)
u = torch.tensor([1.0, 2.0, 3.0])
v = torch.tensor([4.0, -1.0, 0.0])
# You already know u dot v should come out to 2.


# 1) DOT PRODUCT: implement manually, no torch.dot
def my_dot(a, b):
    # TODO: multiply matching components, then sum them.
    # Hint: elementwise multiply is just a * b; summing is .sum()
    return (a * b).sum()


print("my_dot:   ", my_dot(u, v))
print("torch.dot:", torch.dot(u, v))  # should match


my_dot:    tensor(2.)
torch.dot: tensor(2.)


In [3]:
u ** 2

tensor([1., 4., 9.])

In [4]:
# 2) L2 NORM: implement manually, no torch.norm
def my_norm(a):
    # TODO: square each component, sum, square-root.
    # Hint: a ** 2 squares elementwise; torch.sqrt(...) takes the root.
    # Bonus: notice my_norm(a) is just sqrt(my_dot(a, a)).
    return torch.sqrt((a**2).sum())


print("my_norm:   ", my_norm(u))
print("torch.norm:", torch.linalg.norm(u))  # should match



my_norm:    tensor(3.7417)
torch.norm: tensor(3.7417)


In [5]:
# 3) COSINE SIMILARITY: build it from the two functions above
def my_cosine(a, b):
    # TODO: dot product divided by (norm of a * norm of b)
    return torch.dot(a, b) / (torch.linalg.norm(a) * torch.linalg.norm(b))


print("my_cosine:", my_cosine(u, v))
print("torch:    ", torch.nn.functional.cosine_similarity(u, v, dim=0))  # should match




my_cosine: tensor(0.1296)
torch:     tensor(0.1296)


In [6]:
# 4) PROJECTION of v onto u: return the projection VECTOR, the shadow itself
def project(v, u):
    # TODO: scale u by (u dot v) / (u dot u).
    # Think: scalar projection is (u dot v) / |u|; the vector version
    # points along u, so multiply the unit vector u_hat by that scalar.
    # That simplifies to ((u dot v) / (u dot u)) * u. Work out why.
    return u * (torch.dot(u, v) / torch.dot(u, u))

print("projection of v onto u:", project(v, u))
# Not sure about this one

projection of v onto u: tensor([0.1429, 0.2857, 0.4286])


## Projection Verification

There is no single PyTorch built-in you need for projection here, but you can verify it geometrically.

If `p = project(v, u)`, then:

```text
p is parallel to u
v - p is orthogonal to u
dot(v - p, u) should be approximately 0
```

That `v - p` vector is the leftover part after the shadow along `u` has been removed.

API you may need:

```python
p = project(v, u)
leftover = v - p
my_dot(leftover, u)
torch.isclose(value, torch.tensor(0.0))
```

In [8]:
p = project(v, u)
print(p)
# p is parallel to u because projection here is saying how much of v lands on u's direction

tensor([0.1429, 0.2857, 0.4286])


In [11]:
my_dot(v-p, u)

tensor(0.)

In [12]:
my_dot(u, v-p)

tensor(0.)

In [13]:
# zero dot product means orthogonal, they point in independent directions

## By-Hand Projection Checks in PyTorch

Use your `project(v, u)` function to check the projection problems from above:

1. Project `(3, 4)` onto `(1, 0)`.
2. Project `(3, 4)` onto `(1, 1) / sqrt(2)`.

API you may need:

```python
x = torch.tensor([3.0, 4.0])
axis = torch.tensor([1.0, 0.0])
diag = torch.tensor([1.0, 1.0]) / torch.sqrt(torch.tensor(2.0))
project(x, axis)
project(x, diag)
```

In [15]:
x = torch.tensor([3., 4])
axis = torch.tensor([1., 0])
project(x, axis)

tensor([3., 0.])

## Alignment Experiments

Use small vectors to build intuition. For each pair, compute dot product, norms, and cosine similarity. Then write what the result means.

Try these pairs:

```text
same direction:      (1, 0) and (3, 0)
opposite direction:  (1, 0) and (-3, 0)
perpendicular:       (1, 0) and (0, 5)
diagonal-ish:        (1, 1) and (2, 1)
```

Question: when does magnitude change the dot product but not cosine similarity?

In [18]:
x = torch.tensor([1.,0])
y = torch.tensor([3.,0])
print(my_dot(x, y))
print(my_norm(x))
print(my_norm(y))
print(my_cosine(x, y))

tensor(3.)
tensor(1.)
tensor(3.)
tensor(1.)


In [19]:
x = torch.tensor([1.,0])
y = torch.tensor([-3.,0])
print(my_dot(x, y))
print(my_norm(x))
print(my_norm(y))
print(my_cosine(x, y))

tensor(-3.)
tensor(1.)
tensor(3.)
tensor(-1.)


In [22]:
x = torch.tensor([1.,0])
y = torch.tensor([0, 5.])
print(my_dot(x, y))
print(my_norm(x))
print(my_norm(y))
print(my_cosine(x, y))

tensor(0.)
tensor(1.)
tensor(5.)
tensor(0.)


In [23]:
x = torch.tensor([1.,0])
y = torch.tensor([2.,1])
print(my_dot(x, y))
print(my_norm(x))
print(my_norm(y))
print(my_cosine(x, y))

tensor(2.)
tensor(1.)
tensor(2.2361)
tensor(0.8944)


## Bridge to Attention

Attention uses dot products to score alignment. A query vector and a key vector get a high score when they point in compatible directions.

For now, do not worry about full attention. Just connect the ideas:

```text
attention_score = query dot key
```

Later, the model learns transformations that produce query and key vectors. The dot product then measures learned alignment, not just raw geometric similarity.

### Tiny attention-score intuition check

Create one query vector and three key vectors. Compute the dot product between the query and each key. Which key scores highest? Does that match your geometric intuition?

API you may need:

```python
q = torch.tensor([...], dtype=torch.float32)
keys = torch.tensor([...], dtype=torch.float32)  # shape: (num_keys, dim)
scores = keys @ q  # one score per key
torch.argmax(scores)
```

In [27]:
q = torch.tensor([1,1.])
keys = torch.tensor([[3,1.], [-1., -1], [0, 4.]])
scores = keys @ q
print(scores)
torch.argmax(scores)

tensor([ 4., -2.,  4.])


tensor(0)

## Output Note

Write a short note titled: **Dot product as alignment: from geometry to attention**.

Include:

1. What the dot product measures geometrically.
2. Why norms matter.
3. Why cosine similarity removes magnitude.
4. What projection means.
5. How this connects to attention scores.

## What to Submit

Paste back:

1. Your by-hand answers for problems 1-5.
2. Your function bodies for `my_dot`, `my_norm`, `my_cosine`, and `project`.
3. Anything that surprised you about dot product vs cosine similarity.
4. Your short output note.